# Recall-Analyse: Wirkung der IoU-Schwelle

Die Standard-Metrik zählt eine Krone erst ab **IoU ≥ 0.5** als Treffer — für kleine
Baumkronen in Luftbildern ungewöhnlich streng (in der Baumkronen-Literatur ist 0.3–0.4
üblich). Dieses Notebook zeigt, wie viel Recall/F1 diese Schwelle „frisst" und ob die
verpassten Kronen **gar nicht erkannt** oder nur **form-/größenmäßig daneben** sind.

Datenquelle: `iou_sweep.py` (läuft als `nda`, cached die Inferenz einmal, wendet das
getunte Config-PP an) schreibt je Modell:
- `runs/<name>/iou_curve_<res>.csv`      — P/R/F1 über IoU-Schwellen
- `runs/<name>/iou_composition_<res>.csv` — GT-Kronen als zero / partial / hit

Erzeugen (Beispiel):
```
.venv/bin/python 3_Model/src/iou_sweep.py --config 3_Model/configs/finetune_step1_ndom_spring75.yaml \
    --checkpoint 3_Model/runs/step1_ndom_spring75/checkpoints/best.pt --resolution 7.5cm
```


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/home/leafline/leafline/3_Model/runs')

# (Label, run-Verzeichnis, Auflösung) — fehlende werden übersprungen
MODELS = [
    ('step1 (5ch) 7.5cm',        'step1_spring75',      '7.5cm'),
    ('step1+nDOM (6ch) 7.5cm',   'step1_ndom_spring75', '7.5cm'),
    ('step3 (50/50) 20cm-spring','step3_mix20',         '20cm-spring'),
]
curves, comps = {}, {}
for label, run, res in MODELS:
    cp = R/run/f'iou_curve_{res}.csv'; kp = R/run/f'iou_composition_{res}.csv'
    if cp.exists(): curves[label] = pd.read_csv(cp); print('geladen:', label)
    else: print('FEHLT (iou_sweep noch nicht gelaufen):', cp)
    if kp.exists(): comps[label] = pd.read_csv(kp)
if not curves: print('\nNoch keine CSVs — erst iou_sweep.py als nda laufen lassen.')

## 1. Recall & F1 über die IoU-Schwelle

In [ ]:
if curves:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))
    for label, df in curves.items():
        df = df.sort_values('iou_threshold')
        ax1.plot(df['iou_threshold'], df['recall'], marker='o', label=label)
        ax2.plot(df['iou_threshold'], df['f1'],     marker='o', label=label)
    for ax, t in [(ax1,'Recall'),(ax2,'F1')]:
        ax.axvline(0.5, color='gray', ls='--', alpha=0.6)
        ax.axvline(0.3, color='green', ls='--', alpha=0.6)
        ax.set_xlabel('IoU-Schwelle'); ax.set_ylabel(t); ax.set_title(f'{t} vs. IoU-Schwelle')
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    ax1.text(0.31,0.02,'0.3',color='green'); ax1.text(0.51,0.02,'0.5',color='gray')
    plt.tight_layout(); plt.show()

## 2. F1 & Recall: aktuelle Schwelle 0.5 vs. gelockerte 0.3

In [ ]:
if curves:
    def at(df, thr, col):
        row = df[np.isclose(df['iou_threshold'], thr)]
        return round(float(row[col].iloc[0]),3) if len(row) else np.nan
    rows=[]
    for label, df in curves.items():
        rows.append({'Modell':label,
                     'F1@0.5':at(df,0.5,'f1'), 'F1@0.3':at(df,0.3,'f1'),
                     'Recall@0.5':at(df,0.5,'recall'), 'Recall@0.3':at(df,0.3,'recall')})
    t=pd.DataFrame(rows).set_index('Modell')
    t['F1-Faktor (0.3/0.5)']=(t['F1@0.3']/t['F1@0.5']).round(2)
    print(t.to_string())

## 3. Zusammensetzung der GT-Kronen (zero / partial / hit)

In [ ]:
if comps:
    fig, ax = plt.subplots(figsize=(9,0.9*len(comps)+1.5))
    labels=list(comps.keys()); y=np.arange(len(labels))
    cats=['zero','partial','hit']; colors={'zero':'#c0392b','partial':'#e0a800','hit':'#2a9d3a'}
    left=np.zeros(len(labels))
    for cat in cats:
        vals=[float(comps[l][comps[l].category==cat]['pct'].iloc[0]) for l in labels]
        ax.barh(y, vals, left=left, color=colors[cat], label=cat)
        for yi,(v,l) in enumerate(zip(vals,left)):
            if v>4: ax.text(l+v/2, yi, f'{v:.0f}%', va='center', ha='center', fontsize=8, color='white')
        left+=vals
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.set_xlabel('% der GT-Kronen')
    ax.set_title('zero (keine Überlappung) / partial (IoU<0.5) / hit (≥0.5)')
    ax.legend(ncol=3, loc='lower right', fontsize=8); plt.tight_layout(); plt.show()
    print('zero → Nicht-Erkennung (Loss/Sampling) · partial → Form/Größe (PP/dist-Loss/Metrik)')

## 4. Deutung

- **IoU 0.5 ist streng für Baumkronen.** Der Recall/F1 steigt bei 0.3 deutlich (für
  step1+nDOM z. B. F1 0.158 → 0.380). 0.3 ist ein legitimer, in der Literatur üblicher
  Berichtswert — der zusätzlich zu 0.5 dokumentiert werden sollte.
- **Zwei Ursachen, beide groß** (step1+nDOM @7.5cm): ~40% der GT-Kronen haben *keine*
  überlappende Vorhersage (echte Nicht-Erkennung → Loss/Sampling), ~48% überlappen, aber
  unter 0.5 (Form/Größe → PP / dist-Loss / Metrik).
- **Nächster Hebel:** Loss Richtung Recall umgewichten (Tversky, FN stärker bestrafen) —
  adressiert die 40% Nicht-Erkennung und lässt die 48% partial wachsen. Parallel 0.3 als
  zweite Metrik berichten.
